In [2]:
from datasets import load_dataset_builder, load_dataset, get_dataset_split_names
from log_wrapper import log_calls
from huggingface_hub import hf_hub_download
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

import pandas as pd

/Users/vladpalamarchuk/anaconda3/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
ds = load_dataset("yandex/yambda", "flat-multievent-50m", split="train")
df = ds.to_pandas()

# likes_df = pd.read_parquet("hf://datasets/yandex/yambda/flat/50m/likes.parquet")
artist_map = pd.read_parquet("hf://datasets/yandex/yambda/artist_item_mapping.parquet")
album_map = pd.read_parquet("hf://datasets/yandex/yambda/album_item_mapping.parquet")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47790449 entries, 0 to 47790448
Data columns (total 7 columns):
 #   Column                Dtype  
---  ------                -----  
 0   uid                   uint32 
 1   timestamp             uint32 
 2   item_id               uint32 
 3   is_organic            uint8  
 4   played_ratio_pct      float64
 5   track_length_seconds  float64
 6   event_type            object 
dtypes: float64(2), object(1), uint32(3), uint8(1)
memory usage: 1.6+ GB


In [7]:
df['item_id'].nunique()  

934057

In [21]:
df.sample(5)

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
32492767,678000,6100225,6901374,0,100.0,200.0,listen
13283762,276600,6353675,134761,0,99.0,130.0,listen
15891720,332700,24204135,1497131,1,2.0,225.0,listen
31384306,656500,19852140,1127123,1,100.0,185.0,listen
32500935,678100,15459890,2445140,1,100.0,215.0,listen


**Catboost**

In [26]:
threshold_time = df["timestamp"].quantile(0.8)

# Создаём бинарный таргет: был ли лайк на этот трек от этого юзера
liked_pairs = df[df["event_type"] == "like"][["uid", "item_id"]].drop_duplicates()
liked_pairs["liked"] = 1

# Берём только прослушивания
listens = df[df["event_type"] == "listen"].copy()

# Мерджим — если был лайк на эту пару (uid, item_id), то liked=1
listens = listens.merge(liked_pairs, on=["uid", "item_id"], how="left")
listens["liked"] = listens["liked"].fillna(0).astype(int)

X = listens.drop(columns=["event_type", "liked"])
y = listens["liked"]

# Фильтр по индексу
train_mask = listens["timestamp"] <= threshold_time

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

In [ ]:
model = CatBoostClassifier(
    n_estimators=100,
    random_state=42,
    cat_features=["is_organic"],
    auto_class_weights='Balanced'
)
model.fit(X_train, y_train)

In [29]:
# Вероятности нужны для AUC метрик
preds_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, preds_proba))
print("PR-AUC:", average_precision_score(y_test, preds_proba))
print(classification_report(y_test, model.predict(X_test)))

ROC-AUC: 0.780414310097065
PR-AUC: 0.4862402912180316
              precision    recall  f1-score   support

           0       0.93      0.61      0.74   6998111
           1       0.42      0.87      0.57   2287111

    accuracy                           0.67   9285222
   macro avg       0.68      0.74      0.65   9285222
weighted avg       0.81      0.67      0.69   9285222

